In [12]:
import warnings 
warnings.filterwarnings("ignore")
from langchain_community.document_loaders import PyPDFLoader 
loader = PyPDFLoader("economics_research_reference.pdf")
pages = loader.load()

In [13]:
from langchain_text_splitters import RecursiveCharacterTextSplitter 
spliter = RecursiveCharacterTextSplitter(chunk_size=1400 , chunk_overlap=120)
texts = spliter.split_documents(pages)
chunks = [i.page_content for i in texts]
metadata = [i.metadata for i in texts]

import chromadb 
from chromadb.utils.embedding_functions import SentenceTransformerEmbeddingFunction 
embedding = SentenceTransformerEmbeddingFunction()

client = chromadb.PersistentClient(path="./Economics_DB")

collection = client.get_or_create_collection(name="Evelute",embedding_function=embedding)

if collection.count()==0:
    collection.add(
        documents=chunks , 
        ids=[str(i) for i in range(len(chunks))],
        metadatas=metadata
    )

collection.count()

21

In [14]:

from langchain_core.tools import tool 
from langchain_community.tools import DuckDuckGoSearchRun 
import ast 

@tool 
def calculator(exec:str):
    """ Provided Anytypes of arithmetic operations calculate by this Tool """ 
    try :
        return str(ast.literal_eval(exec))
    except Exception as e :
        return str(e)

@tool 
def retrive(query:str):
    """ Provided user query search for answer on the local document then transfer to the websearch [if it cant find out any related content]""" 
    result = collection.query(query_texts=[query],n_results=3)
    dis = result['distances'][0]
    document = result['documents'][0]
    thresold = 0.9
    goo_chun=[docs for i , docs in zip(dis,document) if i<thresold]

    if goo_chun:
        return "\n\n".join(goo_chun)
    search = DuckDuckGoSearchRun()
    result = search.run(query)
    return result  

tools = [calculator,retrive]
tool_name = {t.name:t for t in tools}
tool_name 

{'calculator': StructuredTool(name='calculator', description='Provided Anytypes of arithmetic operations calculate by this Tool', args_schema=<class 'langchain_core.utils.pydantic.calculator'>, func=<function calculator at 0x11f379510>),
 'retrive': StructuredTool(name='retrive', description='Provided user query search for answer on the local document then transfer to the websearch [if it cant find out any related content]', args_schema=<class 'langchain_core.utils.pydantic.retrive'>, func=<function retrive at 0x120f49360>)}

In [15]:
from langchain_groq import ChatGroq 
import os 
from dotenv import load_dotenv 
load_dotenv()
key=os.getenv("GROQ_API_KEY")
chat = ChatGroq(model="llama-3.1-8b-instant")
LLM_bind = chat.bind_tools(tools)

In [16]:
from langchain_core.messages import ToolMessage

def tool_fun(question):
    messages = [{"role": "user", "content": question}]
    response = LLM_bind.invoke(messages)

    if not response.tool_calls:
        return response.content
    retrive_content=[]
    messages.append(response)

    for call in response.tool_calls:
        tool = tool_name[call['name']]
        result = tool.invoke(call['args'])
        retrive_content.append(result)
        messages.append(ToolMessage(content=str(result), tool_call_id=call['id']))

    final = LLM_bind.invoke(messages)
    return final.content ,  retrive_content


query = "what is sales?"
generated_answer = tool_fun(query)
print(generated_answer)

('', ["and income changes.\nPerfect competition\nPrice-taking firms produce where marginal cost equals price; free entry drives long-run economic profit to zero.\nMonopoly\nA single price-searching firm sets marginal revenue equal to marginal cost, producing less and charging more\nthan the competitive benchmark.\nIS-LM\nGoods-market and money-market equilibrium loci jointly determine output and the interest rate in the short run\nunder sticky prices.\nSolow growth model\nCapital accumulation, depreciation, and population growth determine a steady state; technology shifts the\nsteady state itself.\n\nmust ask who changes behavior, over what horizon, with what information, under which incentives, and with\nwhat distributional consequences. Policy conclusions are strongest when theory, data, institutional detail, and\nrobustness checks point in the same direction.\n4. Firm Behavior and Market Structure\nProfit maximization\nA firm maximizes profit by producing the quantity at which margi

In [18]:
from pydantic import BaseModel, Field
from typing import Literal 

class Build(BaseModel):
    reasoning: str = Field(description="A one-line explanation of why this verdict was chosen.")
    verdict: Literal["fully_grounded", "partial_grounded", "hallucinated"]

def judge(query: str, context :str , answer: str):
    prompt = f"""You are a strict RAG judge. Judge the generated answer based ONLY on the provided query.

Query: {query}
Context: {context}
Answer: {answer}

Instructions for Verdict:
- fully_grounded: answer is entirely supported by the context.
- partial_grounded: answer relates to the context but adds unsupported detail.
- hallucinated: answer comes from memory, not the context (or context is empty/None).

if user asking any question's answers find out in the local document then direct give answer and if dont then directly say "NOT RELATED CONTENT"

"""

    evaluator = chat.with_structured_output(Build)
    return evaluator.invoke(prompt)

queries = [
    "what is Demand ?",
    "what is 12*8?", 
    "what is the largest country in the world right now ?"
]
for qustion in queries:
    content , generated_answer = tool_fun(qustion)   
    content = content 
    generated_answer=generated_answer
    print(f'the agent output : {content}')
    evelute = judge(query=qustion , context=content, answer=generated_answer)
    print(f'the reaseon : {evelute.reasoning}')
    print(f'the vedict : {evelute.verdict}')

the agent output : 
the reaseon : The provided answer directly relates to the topic of demand, and its explanation is entirely supported by the given text, specifically the sections on opportunity cost, production possibilities frontier, and supply and demand schedules.
the vedict : fully_grounded
the agent output : 
the reaseon : The answer 'malformed node or string on line 1: <ast.BinOp object at 0x120e1c250>' does not relate to the context of the math problem 12*8.
the vedict : hallucinated
the agent output : 
the reaseon : The generated answer does not relate to the query at all.
the vedict : hallucinated
